In [528]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from typing import List
from tqdm import tqdm
import random
import seaborn as sns
from typing import Tuple
import pandas as pd

In [529]:
df_users = pd.read_csv('KION_DATASET/data_original/users.csv')
df_interactions = pd.read_csv('KION_DATASET/interactions.csv')
df_items = pd.read_csv('KION_DATASET/data_original/items.csv')

In [530]:
df_interactions.head()

,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250.0,72.0
1,699317,1659,2021-05-29,8317.0,100.0
2,656683,7107,2021-05-09,10.0,0.0
3,864613,7638,2021-07-05,14483.0,100.0
4,964868,9506,2021-04-30,6725.0,100.0


In [531]:
df_users.head()

,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,М,1
1,962099,age_18_24,income_20_40,М,0
2,1047345,age_45_54,income_40_60,Ж,0
3,721985,age_45_54,income_20_40,Ж,0
4,704055,age_35_44,income_60_90,Ж,0


In [532]:
df_items.head()

,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."
2,10716,film,Тактическая сила,Tactical Force,2011.0,"криминал, зарубежные, триллеры, боевики, комедии",Канада,NaN,16.0,NaN,Адам П. Калтраро,"Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Профессиональный рестлер Стив Остин («Все или ...,"Тактическая, сила, 2011, Канада, бандиты, ганг..."
3,7868,film,45 лет,45 Years,2015.0,"драмы, зарубежные, мелодрамы",Великобритания,NaN,16.0,NaN,Эндрю Хэй,"Александра Риддлстон-Барретт, Джеральдин Джейм...","Шарлотта Рэмплинг, Том Кортни, Джеральдин Джей...","45, лет, 2015, Великобритания, брак, жизнь, лю..."
4,16268,film,Все решает мгновение,NaN,1978.0,"драмы, спорт, советские, мелодрамы",СССР,NaN,12.0,Ленфильм,Виктор Садовский,"Александр Абдулов, Александр Демьяненко, Алекс...",Расчетливая чаровница из советского кинохита «...,"Все, решает, мгновение, 1978, СССР, сильные, ж..."


In [533]:
df_interactions['last_watch_dt'].unique()
df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'], errors='coerce')
df_interactions = df_interactions.dropna(subset=['last_watch_dt'])

In [534]:
old_item = df_items[df_items['release_year'] > 1960.0] # уберем старые фильмы до 1960 года
df_interactions = df_interactions[df_interactions['item_id'].isin(old_item['item_id'])]
df_interactions = df_interactions.drop(['total_dur'], axis=1, inplace=False) # уберем время просмотра

In [535]:
# from datetime import datetime, timedelta
# test_size_days = 8
# size_test = 1 # в test берем последнюю неделю
# data = df_interactions[(df_interactions['watched_pct']>50.0)] # уберем фильмы с просмотром менее 25%
# items_data = data['item_id'].value_counts()
# active_items = items_data[items_data > 5].index # уберем фильмы с которыми взаимодействовали менее 5 раз
# data = data[data['item_id'].isin(active_items)]
# data_train = data[data['last_watch_dt'] < data['last_watch_dt'].max() - timedelta(days=test_size_days)]
# data_test = data[data['last_watch_dt'] >= data['last_watch_dt'].max() - timedelta(days=test_size_days)]
# # data_train = data[data['last_watch_dt'] < data['last_watch_dt'].max() - timedelta(weeks=size_test)]
# # data_test = data[data['last_watch_dt'] >= data['last_watch_dt'].max() - timedelta(weeks=size_test)]

In [536]:
from datetime import datetime, timedelta
test_size_days = 10

# Тестовый промежуток времени равен 10 дней
max_date = df_interactions['last_watch_dt'].max()
test_start = max_date - timedelta(days=test_size_days)

df_interactions = df_interactions[(df_interactions['watched_pct']>50.0)] # уберем фильмы с просмотром менее 50%
df_interactions_train = df_interactions[df_interactions['last_watch_dt'] < test_start]
df_interactions_test = df_interactions[df_interactions['last_watch_dt'] >= test_start]

# df_interactions_train = df_interactions[df_interactions['last_watch_dt'] < df_interactions['last_watch_dt'].max() - timedelta(days=test_size_days)]
# df_interactions_test = df_interactions[df_interactions['last_watch_dt'] >= df_interactions['last_watch_dt'].max() - timedelta(days=test_size_days)]

In [537]:
train_sorted = df_interactions_train.sort_values(['user_id', 'last_watch_dt'])
    
train = train_sorted.groupby('user_id').tail(15) # у пользователей в train берем последние 15 фильмов

In [538]:
test_users = set(df_interactions_test['user_id'].unique())
train_users = set(train['user_id'].unique())
common_users = test_users & train_users
    
print(f"Пользователей в test: {len(test_users)}")
print(f"Пользователей в train: {len(train_users)}")
print(f"Общих пользователей: {len(common_users)}")

Пользователей в test: 47347
Пользователей в train: 281756
Общих пользователей: 21391


In [539]:
def get_active_users_ids(df, min_interactions=2):
    """Возвращает ID пользователей, у которых больше минимального числа взаимодействий."""
    counts = df['user_id'].value_counts()
    return set(counts[counts >= min_interactions].index)

# 1. Находим активных пользователей в каждом наборе данных
active_train_ids = get_active_users_ids(train)
active_test_ids = get_active_users_ids(df_interactions_test)

# 2. Находим пересечение (пользователи, активные и в train, и в test)
common_active_users = active_train_ids & active_test_ids

# 3. Фильтруем исходные данные, оставляя только общих активных пользователей
df_interactions_train = train[train['user_id'].isin(common_active_users)]
df_interactions_test = df_interactions_test[df_interactions_test['user_id'].isin(common_active_users)]

In [540]:
# # Функция для получения индексов активных пользователей
# get_active = lambda df: df['user_id'].value_counts().loc[lambda x: x > 1].index

# # Получаем списки активных пользователей
# train_active_users = set(get_active(train))
# test_active_users = set(get_active(df_interactions_test))

# # Оставляем только пересечение пользователей в обоих датасетах
# common_users = train_active_users & test_active_users

# df_interactions_train = train[train['user_id'].isin(common_users)]
# df_interactions_test = df_interactions_test[df_interactions_test['user_id'].isin(common_users)]

In [541]:
test_users = set(df_interactions_test['user_id'].unique())
train_users = set(df_interactions_train['user_id'].unique())
common_users = test_users & train_users
    
print(f"Пользователей в test: {len(test_users)}")
print(f"Пользователей в train: {len(train_users)}")
print(f"Общих пользователей: {len(common_users)}")

Пользователей в test: 5019
Пользователей в train: 5019
Общих пользователей: 5019


In [542]:
def precision(recommended_list, bought_list):
    recommended = np.array(recommended_list)
    bought = np.array(bought_list)

    # флаги: какие рекомендованные товары действительно куплены
    flags = np.isin(recommended, bought)

    return flags.sum() / len(recommended)


def precision_at_k(recommended_list, bought_list, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)

    flags = np.isin(recommended, bought)

    return flags.sum() / k


def money_precision_at_k(recommended_list, bought_list, prices_recommended, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)
    prices = np.array(prices_recommended[:k])

    # флаги: куплен ли товар
    flags = np.isin(recommended, bought)

    # учитываем деньги
    return np.dot(flags, prices) / prices.sum()

In [543]:
def recall(recommended_list, bought_list):
    recommended = np.array(recommended_list)
    bought = np.array(bought_list)

    # какие купленные товары были среди рекомендованных
    flags = np.isin(bought, recommended)

    return flags.sum() / len(bought)


def recall_at_k(recommended_list, bought_list, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)

    flags = np.isin(bought, recommended)

    return flags.sum() / len(bought)


def money_recall_at_k(recommended_list, bought_list, prices_recommended, prices_bought, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)
    prices_bought = np.array(prices_bought)

    # флаги: купленный товар есть в топ-k рекомендациях
    flags = np.isin(bought, recommended)

    # учитываем деньги (важны цены купленных товаров)
    return np.dot(flags, prices_bought) / prices_bought.sum()

In [544]:
def ap_k(recommended_list, bought_list, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)

    # релевантность: рекомендованный товар куплен или нет
    flags = np.isin(recommended, bought)

    # если нет ни одного релевантного — AP = 0
    if flags.sum() == 0:
        return 0

    sum_precision = 0

    for i in range(k):
        if flags[i]:
            # precision@i+1 (на префиксе)
            precision_i = np.isin(recommended[:i+1], bought).sum() / (i + 1)
            sum_precision += precision_i

    # нормируем на число релевантных объектов в топ-k
    return sum_precision / flags.sum()


def map_k(recommended_list, bought_list, k=5, u=1):
    
    # your_code
    if u == 1:
        return ap_k(recommended_list[u-1], bought_list[u-1], k=5)
    
    sum = 0
    for i in range(0, u):
        ap_k_map = ap_k(recommended_list[i], bought_list[i], k=5)
        sum += ap_k_map

    result = sum / u
    
    return result

In [545]:
data_test_group = df_interactions_test.groupby(df_interactions_test['user_id'])['item_id'].unique().reset_index()
data_test_group.columns = ['user_id', 'item_id']
data_test_group

,user_id,item_id
0,259,"[10509, 10772]"
1,655,"[11188, 5199]"
2,753,"[9327, 8350]"
3,835,"[5434, 11640, 10878]"
4,960,"[8636, 12770, 6809]"
...,...,...
5014,1096841,"[10219, 3620]"
5015,1097296,"[3006, 16087]"
5016,1097444,"[13650, 12841, 13099, 1124, 12250, 2483]"
5017,1097459,"[11844, 7793]"


In [546]:
userids = data_test_group['user_id'].values
 
userids_test = np.arange(len(userids))

In [547]:
# users_unique_test = data_test_group['user_id']
# items_unique = df_interactions['item_id'].unique()

# size_recommend = 10

# recommendations = []
# for user in users_unique_test:
#     random_items = np.random.choice(items_unique, size=size_recommend, replace=False)
#     recommendations.append(random_items)

# result_random = data_test_group.copy()
# result_random['random_recommendation'] = recommendations

In [548]:
def generate_random_recommendations(df_users, df_items, top_n=10, seed=42):
    """
    Генерирует случайные рекомендации для уникальных пользователей.
    """
    np.random.seed(seed)
    
    # 1. Получаем уникальных пользователей и все доступные товары
    unique_users = df_users['user_id'].unique()
    all_items = df_items['item_id'].unique()
    
    # 2. Защита: если товаров меньше, чем нужно рекомендовать
    k = min(top_n, len(all_items))
    
    # 3. Генерируем рекомендации в виде словаря {user_id: [items]}
    # Это быстрее и безопаснее, чем цикл с append
    recommendations_map = {
        user: np.random.choice(all_items, size=k, replace=False).tolist()
        for user in unique_users
    }
    
    # 4. Создаем копию датафрейма и мапим рекомендации по user_id
    result = df_users.copy()
    result['random_recommendation'] = result['user_id'].map(recommendations_map)
    
    return result

# Использование
result_random = generate_random_recommendations(
    df_users=data_test_group, 
    df_items=df_interactions, 
    top_n=10
)

In [549]:
# import numpy as np

# np.random.seed(42)

# unique_users = data_test_group['user_id'].unique()
# all_items = df_interactions['item_id'].unique()
# k = min(10, len(all_items))

# # Создаем список словарей для быстрого создания DataFrame
# rec_data = []
# for user in unique_users:
#     items = np.random.choice(all_items, size=k, replace=False)
#     for item in items:
#         rec_data.append({'user_id': user, 'recommended_item': item})

# df_recommendations = pd.DataFrame(rec_data)

# # Если нужно объединить с исходным тестом (например, для слияния по user_id)
# result_random = data_test_group.merge(df_recommendations, on='user_id', how='left')

In [550]:
def getRandomRecommend(items, k=10):
  return np.random.choice(items, k)

In [551]:
def getFirstK(items, k=10):
  return items[:k]

In [555]:
# items взять из тестового набора



print('random map_k =', map_k(result_random['random_recommendation'], result_random['item_id'], k=10, u=len(result_random)))


print(map_k)

random map_k = 0.0005279936242279339
<function map_k at 0x00000203435B4CA0>


In [562]:
# def popularity(train, n=10):
#     train_group = train.groupby(train['item_id'])['user_id'].unique().reset_index()
#     train_group.columns = ['item_id', 'user_id']

#     train_group['user_sum'] = train_group['user_id'].apply(len)
#     top_10_train = train_group.nlargest(n, 'user_sum')
#     top_10 = top_10_train['item_id']

#     return top_10.tolist()

# popular = popularity(df_interactions_train, n=10)
# result_popular = data_test_group.copy()
# result_popular['popular_recommendation'] = result_popular['user_id'].apply(lambda x: popular)

In [564]:
def get_popular_items(train, n=10):
    """Возвращает список из n самых популярных товаров."""
    top_items = train['item_id'].value_counts().nlargest(n)
    return top_items.index.tolist()

# 1. Получаем список популярных товаров (длиной 10)
popular = get_popular_items(df_interactions_train, n=10)

# 2. Присваиваем рекомендации
result_popular = data_test_group.copy()

# ВАРИАНТ А: Список списков (Рекомендуемый)
# Создаем список, где каждый элемент — это копия списка popular.
# Длина этого нового списка будет равна количеству строк в датафрейме.
result_popular['popular_recommendation'] = [popular] * len(result_popular)

In [565]:


result_popular



,user_id,item_id,popular_recommendation
0,259,"[10509, 10772]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."
1,655,"[11188, 5199]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."
2,753,"[9327, 8350]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."
3,835,"[5434, 11640, 10878]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."
4,960,"[8636, 12770, 6809]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."
...,...,...,...
5014,1096841,"[10219, 3620]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."
5015,1097296,"[3006, 16087]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."
5016,1097444,"[13650, 12841, 13099, 1124, 12250, 2483]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."
5017,1097459,"[11844, 7793]","[9728, 13865, 3734, 15297, 10440, 142, 8636, 7..."


In [566]:


print('popular map_k =', map_k(result_popular['popular_recommendation'], result_popular['item_id'], k=10, u=len(result_popular)))



popular map_k = 0.060815567510128114
